In [2]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import re
from src.db import DATA_RAW

df = pd.read_parquet(DATA_RAW / "cars.parquet")
text = df["seller_description"].fillna("").str.lower()

KEYWORDS = {
    "kw_good_state":    r"(идеальн|отличн|хорош|техническ)[а-яё]*\s+состоян",
    "kw_dealer_serv":   r"официальн[а-яё]*\s+дилер",
    "kw_max_trim":      r"максимальн[а-яё]*\s+комплектац",
    "kw_any_check":     r"любы[а-яё]*\s+проверк",
    "kw_no_dtp":        r"без\s+дтп|не\s+бит|не\s+крашен",
    "kw_one_owner":     r"один\s+владел|1\s+владел|один\s+хозяин",
    "kw_serviced":      r"полностью\s+обслужен|замена\s+масла|сервисн[а-яё]*\s+книж",
    "kw_no_complaints": r"без\s+нарекан",
    "kw_sel_poehal":    r"сел\s+(и\s+)?поехал",
}

for name, pat in KEYWORDS.items():
    df[name] = text.str.contains(pat, regex=True, na=False)
    print(f"{name:20s} {df[name].sum():6d}  ({df[name].mean()*100:.1f}%)")

C:\Users\Андрей\AppData\Local\Temp\ipykernel_14880\616491887.py:26: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[name] = text.str.contains(pat, regex=True, na=False)


kw_good_state         12515  (41.5%)
kw_dealer_serv         5316  (17.6%)
kw_max_trim            1218  (4.0%)
kw_any_check           1438  (4.8%)
kw_no_dtp              2172  (7.2%)
kw_one_owner           3390  (11.2%)
kw_serviced            2640  (8.8%)
kw_no_complaints        770  (2.6%)
kw_sel_poehal           865  (2.9%)


C:\Users\Андрей\AppData\Local\Temp\ipykernel_14880\616491887.py:26: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df[name] = text.str.contains(pat, regex=True, na=False)


In [3]:
sample = 'Авто без вложений сел - поехал.'
import re
print(re.findall(r"сел\s*[-–—]?\s*(и\s+)?поехал", sample.lower()))
# должно быть [' - '] или ['']

['']


In [4]:
brands_to_check = ["Лада", "Kia", "Hyundai", "Toyota", "BMW", "Mercedes-Benz",
                   "Volkswagen", "Nissan", "Skoda", "Ford"]

for flag in ["kw_good_state", "kw_dealer_serv", "kw_max_trim", "kw_any_check"]:
    print(f"\n=== {flag} ===")
    for brand in brands_to_check:
        sub = df[df["brand"] == brand]
        if sub[flag].sum() < 30:
            continue
        t = sub.loc[sub[flag], "price"].median()
        f = sub.loc[~sub[flag], "price"].median()
        print(f"  {brand:15s} n={sub[flag].sum():4d}  "
              f"T={t:>10,.0f}  F={f:>10,.0f}  lift={t/f:.3f}")


=== kw_good_state ===
  Лада            n= 554  T=   650,000  F=   600,000  lift=1.083
  Kia             n=1008  T= 1,500,000  F= 1,590,000  lift=0.943
  Hyundai         n= 701  T= 1,470,000  F= 1,565,000  lift=0.939
  Toyota          n= 723  T= 2,450,000  F= 2,600,000  lift=0.942
  BMW             n=1029  T= 3,500,000  F= 4,750,000  lift=0.737
  Mercedes-Benz   n=1081  T= 3,784,000  F= 4,500,000  lift=0.841
  Volkswagen      n= 660  T= 1,549,500  F= 1,565,444  lift=0.990
  Nissan          n= 541  T= 1,300,000  F= 1,250,000  lift=1.040
  Skoda           n= 462  T= 1,298,000  F= 1,350,000  lift=0.961
  Ford            n= 392  T= 1,000,000  F= 1,000,000  lift=1.000

=== kw_dealer_serv ===
  Лада            n= 168  T= 1,014,000  F=   580,000  lift=1.748
  Kia             n= 329  T= 1,580,000  F= 1,520,000  lift=1.039
  Hyundai         n= 202  T= 1,575,500  F= 1,500,000  lift=1.050
  Toyota          n= 223  T= 2,990,000  F= 2,427,500  lift=1.232
  BMW             n= 361  T= 4,550,000  F= 

In [5]:
sample = 'Авто без вложений сел - поехал.'
s = sample.lower()
for i, c in enumerate(s):
    if i >= 15 and i <= 25:
        print(i, repr(c), hex(ord(c)))

15 'и' 0x438
16 'й' 0x439
17 ' ' 0x20
18 'с' 0x441
19 'е' 0x435
20 'л' 0x43b
21 ' ' 0x20
22 '-' 0x2d
23 ' ' 0x20
24 'п' 0x43f
25 'о' 0x43e


In [6]:
FINAL_KEYWORDS = {
    "kw_no_dtp":        r"без\s+дтп|не\s+бит|не\s+крашен|не\s+бита",
    "kw_one_owner":     r"один\s+владел|1\s+владел|один\s+хозяин",
    "kw_dealer_serv":   r"официальн[а-яё]*\s+дилер",
    "kw_max_trim":      r"максимальн[а-яё]*\s+комплектац",
    "kw_serviced":      r"полностью\s+обслужен|замена\s+масла|сервисн[а-яё]*\s+книж",
    "kw_no_complaints": r"без\s+нарекан",
    "kw_sel_poehal":    r"сел\s*.{0,3}\s*поехал",
}

text = df["seller_description"].fillna("").str.lower()
for name, pat in FINAL_KEYWORDS.items():
    df[name] = text.str.contains(pat, regex=True, na=False)
    print(f"{name:20s} {df[name].sum():6d}  ({df[name].mean()*100:.1f}%)")

kw_no_dtp              2172  (7.2%)
kw_one_owner           3390  (11.2%)
kw_dealer_serv         5316  (17.6%)
kw_max_trim            1218  (4.0%)
kw_serviced            2640  (8.8%)
kw_no_complaints        770  (2.6%)
kw_sel_poehal           940  (3.1%)
